# Data Preparation Day-1

Agenda Hari 1:
- Sesi 1: Why Data Preparation Matters
- Sesi 2: Acquiring Data
- Sesi 3: EDA — Kenali Datamu
- Sesi 4: Data Cleansing
- Sesi 5: Advanced Imputation
- Rekap Hari 1 + Q&A

## Why data preparation matters?


### CRISP DM Framework

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modelling
5. Evaluation
6. Deployment

## Acquiring Data

In [ ]:
import pandas as pd

# CSV 
df = pd.read_csv('data/data.csv')

# Dengan parameter penting:
# df = pd.read_csv('data/data.csv',
#     sep=',',
#     encoding='utf-8',        # ganti 'latin-1' jika error
#     na_values=[' ', 'N/A', '-'],
#     parse_dates=['tanggal'])

# Excel
# df = pd.read_excel('data/data.xlsx', sheet_name='Realisasi')

# JSON
# df = pd.read_json('data/data.json')

# Parquet (modern, 5-10x lebih cepat dari CSV)
# df = pd.read_parquet('data/data.parquet')

# Remote database
# from sqlalchemy import create_engine
# from dotenv import load_dotenv
# import os
#
# load_dotenv()
#
# conn_str = os.getenv('CONN_STR').strip()
# engine = create_engine(conn_str)
# df = pd.read_sql("SELECT * FROM public.data", engine)

# API -JSON
# import requests
# url = "https://training.ekos.my.id/data"
# response = requests.get(url)
#
# if response.status_code == 200:
#     data = response.json().get('data')
#     df = pd.DataFrame(data)

df

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# Load dataset kotor
telco_data = "https://raw.githubusercontent.com/ekosup/data-preparation/refs/heads/main/data/telco_raw.csv"
df_dirty = pd.read_csv(telco_data)

# # Model dengan data kotor — minimal preprocessing
# df_dirty['TotalCharges'] = pd.to_numeric(df_dirty['TotalCharges'], errors='coerce')
# df_dirty = df_dirty.fillna(0)
# df_dirty['Churn'] = df_dirty['Churn'].map({'Stayed': 0, 'Churned': 1})
#
# df_dirty = df_dirty.dropna(subset=['Churn'])
# df_dirty['Churn'] = df_dirty['Churn'].astype(int)
#
# # Ambil hanya kolom numerik dulu untuk demo cepat
# num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
# X_dirty = df_dirty[num_cols]
# y = df_dirty['Churn']
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X_dirty, y, test_size=0.2, random_state=42, stratify=y)
#
# model_dirty = LogisticRegression(max_iter=1000)
# model_dirty.fit(X_train, y_train)
# print('=== MODEL DATA KOTOR ===')
# print(classification_report(y_test, model_dirty.predict(X_test)))
#
# # Kita akan revisit ini di akhir Hari 2 setelah data bersih
# # dan lihat perbedaan hasilnya!


In [ ]:
df_dirty.columns

In [ ]:
import seaborn as sns
import missingno as msno

# Distribusi kolom numerik
df_dirty.select_dtypes('number').hist(bins=30)

# Boxplot deteksi outlier
sns.boxplot(y=df_dirty['MonthlyCharges'])

# Missing values map
msno.matrix(df_dirty)
msno.heatmap(df_dirty)  # korelasi missing

In [ ]:
# Churn proportion (target variable)
df_dirty['Churn'].value_counts(normalize=True).round(3)
# No: 0.735 Yes: 0.266
# ⚠ Imbalanced!

In [ ]:
df_dirty['TotalCharges']

In [ ]:
import pandas as pd

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Fix tipe data
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

    # 2. Drop NA hanya jika masih ada
    df = df.dropna(subset=['TotalCharges'])

    # 3. Impute numerik (median)
    if 'MonthlyCharges' in df.columns:
        median_val = df['MonthlyCharges'].median()
        df['MonthlyCharges'] = df['MonthlyCharges'].fillna(median_val)

    # 4. Impute kategorikal (modus)
    if 'MultipleLines' in df.columns:
        if not df['MultipleLines'].mode().empty:
            mode_val = df['MultipleLines'].mode()[0]
            df['MultipleLines'] = df['MultipleLines'].fillna(mode_val)

    # 5. Final check
    if df.isnull().sum().sum() != 0:
        raise ValueError("Masih ada missing values!")

    return df

clean_df(df_dirty)

In [ ]:
df = clean_df(df_dirty)

df.head()

In [ ]:
# Metode 1: IQR
Q1 = df['MonthlyCharges'].quantile(0.25)
Q3 = df['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Metode 2: Z-Score
from scipy import stats
z = abs(stats.zscore(df['tenure']))
outliers = df[z > 3]

# Winsorizing (capping) — ganti dengan batas IQR
df['charges_capped'] = df['MonthlyCharges'].clip(lower=lower, upper=upper)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot pertama
sns.boxplot(y=df['MonthlyCharges'], ax=axes[0])
axes[1].set_title('Monthly Charges - Before Winsorizing')

# Plot kedua
sns.boxplot(y=df['charges_capped'], ax=axes[1])
axes[0].set_title('Charges Capped - After Winsorizing')

plt.tight_layout()
plt.show()